# Piano B — Notebook G: REVE-Base (riproduzione su FACED e tutti i protocolli di CBraMod)

Sessione 02, Parte C. Prima il Gate R (R1: solo linear probing; R2: LP+FT, 5 seed, BA pooled 0.548-0.581),
poi i blocchi nello stesso ordine di priorità del prompt: leave-clip-out, permutati, parità di dati, bande,
canali, sonda della clip, valenza binaria, R3 e opzionali. Coda lunga: 2-3 sessioni con la ripresa.

**Impostazioni:** Accelerator **GPU T4 x2**, **Internet ON** (serve per scaricare i pesi), Kaggle Secret
`HF_TOKEN` (token Hugging Face dell'account che ha accettato le condizioni di `brain-bzh/reve-base`).
Input: `pianob-faced-data`, `pianob-code` (+ l'output della sessione precedente di questo notebook).

I pesi si scaricano in /tmp (fuori dagli output della sessione): non vanno ridistribuiti.

In [ ]:
# Setup: trova il codice (Dataset "pianob-code"), lo copia fuori da /kaggle/working (così non finisce
# negli output) e lo aggiunge a sys.path. Non serve modificare nulla qui.
import glob, os, shutil, sys, time
os.environ.setdefault("PIANOB_T0", str(time.time()))      # inizio della sessione (guardia delle 12 h)
os.environ.setdefault("PIANOB_BUDGET_H", "11.3")           # stop pulito prima del limite Kaggle
SYNTHETIC = os.environ.get("PIANOB_SYNTHETIC") == "1"      # solo per prove locali


def _find_code():
    cands = []
    roots = ["/kaggle/input"] if os.path.exists("/kaggle/input") else []
    roots += [os.getcwd(), os.path.dirname(os.getcwd())]
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "pianob", "__init__.py"), recursive=True):
            base = os.path.dirname(os.path.dirname(p))
            if "pianob_state" not in base:
                cands.append(base)
    assert cands, "Codice non trovato: aggiungi il Dataset 'pianob-code' come input (Add Input)."
    # se ci sono più copie, usa quella con VERSION più recente
    def _ver(b):
        v = os.path.join(b, "VERSION")
        return open(v).read().strip() if os.path.exists(v) else ""
    return sorted(cands, key=_ver)[-1]


CODE_SRC = _find_code()
if os.path.exists("/kaggle/working"):
    CODE = "/tmp/pianob_code"
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(CODE_SRC, CODE, ignore=shutil.ignore_patterns("notebooks", "*.zip", "local_work"))
else:
    CODE = CODE_SRC
sys.path.insert(0, CODE)
os.chdir(CODE)
print("codice da", CODE_SRC, "| versione", open(os.path.join(CODE, "VERSION")).read().strip()
      if os.path.exists(os.path.join(CODE, "VERSION")) else "(locale)")

from pianob import kaggle_env, nb_helpers  # noqa: E402
nb_helpers.print_env()
kaggle_env.restore_state_from_inputs()

In [ ]:
# Pesi REVE: token dal Kaggle Secret, revisione risolta e annotata (fissabile con PIANOB_REVE_REVISION)
from pianob.models.reve import weights as reve_weights
if SYNTHETIC:
    print("sintetico: niente download, pesi casuali")
else:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    REVE_INFO = reve_weights.download(HF_TOKEN)
    print({k: v for k, v in REVE_INFO.items() if k != "config"})

In [ ]:
# Controlli obbligatori (Appendice R §R10): conteggi, forme, parametri allenabili, strict, niente NaN
import torch
from pianob.data.adapters import FacedAdapter
from pianob.models.reve.checks import preflight
PREFLIGHT = preflight(FacedAdapter(synthetic=SYNTHETIC).load(), device="cuda" if torch.cuda.is_available() else "cpu",
                      require_weights=not SYNTHETIC)

In [ ]:
ACCOUNT = "nbG_reve"
nb_helpers.run_queue("G", gpus="auto", synthetic=SYNTHETIC)
nb_helpers.queue_status("G")

In [ ]:
nb_helpers.finish(ACCOUNT)